## Loading preprocessed delta file and creating dim_Store

In [0]:
# Importing libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id

In [0]:
#Initializing spark session
spark = SparkSession.builder.appName("dimStore").getOrCreate()

In [0]:
# File location and type
file_location = "/FileStore/tables/Fact_Sales_Preprocessed"
file_type = "delta"

# The applied options are for delta files. For other file types, these will be ignored.
df_store = spark.read.format(file_type) \
  .option("inferSchema", True) \
  .option("header", True) \
  .load(file_location)


In [0]:
# Selecting distinct values for "StoreRegion", "StoreName", "StoreType"
df_store_final = df_store.select("StoreRegion", "StoreName", "StoreType").distinct()

In [0]:
#Skipping default value and creating surrogate key as we have already handled null values

 #as monotonically_increasing_id() starts with 0, we do monotonically_increasing_id()+1
df_store_final = df_store_final.withColumn("Store_Id", monotonically_increasing_id()+1) \
    .select("Store_Id","StoreRegion", "StoreName", "StoreType")


In [0]:
df_store_final.display()

Store_Id,StoreRegion,StoreName,StoreType
1,North,StoreY,Outlet
2,South,StoreY,Retail
3,East,StoreX,Retail
4,North,StoreX,Outlet
5,North,StoreZ,Retail
6,South,StoreZ,Franchise
7,South,StoreZ,Retail
8,East,StoreX,Franchise
9,South,StoreX,Outlet
10,East,StoreZ,Franchise


In [0]:
df_store_final.write \
    .format("delta") \
        .mode("overwrite")\
            .save("/FileStore/tables/DIM_Store")